In [ ]:
#!pip install qiskit qiskit-optimization qiskit-algorithms

In [ ]:
# import warnings
# from scipy.sparse import SparseEfficiencyWarning

# warnings.filterwarnings(
#     "ignore",
#     category=SparseEfficiencyWarning
# )

In [5]:
import itertools
import numpy as np
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler

def validate_qubo_matrix(Q):
    Q = np.asarray(Q, dtype=float)

    if Q.ndim != 2:
        raise ValueError("Q must be a two-dimensional matrix.")

    if Q.shape[0] != Q.shape[1]:
        raise ValueError("Q must be square.")

    if not np.all(np.isfinite(Q)):
        raise ValueError("Q must contain only finite real numbers.")

    return Q

def qubo_energy(Q, x):
    x = np.asarray(x, dtype=int)
    return float(x @ Q @ x)

def qubo_matrix_to_quadratic_program(Q, name="qubo_problem"):
    n = Q.shape[0]
    qp = QuadraticProgram(name)

    for i in range(n):
        qp.binary_var(name=f"x_{i}")

    linear = {}
    quadratic = {}

    for i in range(n):
        if abs(Q[i, i]) > 1e-12:
            linear[f"x_{i}"] = Q[i, i]

    for i in range(n):
        for j in range(i + 1, n):
            coeff = Q[i, j] + Q[j, i]
            if abs(coeff) > 1e-12:
                quadratic[(f"x_{i}", f"x_{j}")] = coeff

    qp.minimize(linear=linear, quadratic=quadratic)

    return qp

def brute_force_qubo(Q):
    n = Q.shape[0]
    best_x = None
    best_energy = np.inf

    for bits in itertools.product([0, 1], repeat=n):
        x = np.array(bits, dtype=int)
        energy = qubo_energy(Q, x)

        if energy < best_energy:
            best_energy = energy
            best_x = x.copy()

    return best_x, best_energy

def solve_qubo_with_qaoa(Q, reps=1, maxiter=200, seed=123):
    Q = validate_qubo_matrix(Q)
    
    qp = qubo_matrix_to_quadratic_program(Q)
    ising_operator, offset = qp.to_ising()

    sampler = StatevectorSampler(seed=seed)
    optimizer = COBYLA(maxiter=maxiter)

    qaoa = QAOA(
        sampler=sampler,
        optimizer=optimizer,
        reps=reps,
    )

    qaoa_optimizer = MinimumEigenOptimizer(qaoa)
    result = qaoa_optimizer.solve(qp)

    x_qaoa = np.array(result.x, dtype=int)
    e_qaoa = qubo_energy(Q, x_qaoa)

    return {
        "solution": x_qaoa,
        "qubo_energy": e_qaoa,
        "qiskit_objective": float(result.fval),
        "ising_operator": ising_operator,
        "offset": offset,
        "raw_result": result,
    }

In [6]:
Q = np.array([
        [1, -2,  0],
        [0,  1, -2],
        [0,  0,  1],
    ], dtype=float)

exact_x, exact_energy = brute_force_qubo(Q)

qaoa_out = solve_qubo_with_qaoa(
        Q=Q,
        reps=1,
        maxiter=200,
        seed=123,
    )

print("Exact solution:", exact_x)
print("Exact energy:", exact_energy)

print("QAOA solution:", qaoa_out["solution"])
print("QAOA QUBO energy:", qaoa_out["qubo_energy"])

print("Energy match:")
print(np.isclose(exact_energy, qaoa_out["qubo_energy"], atol=1e-9))

Exact solution: [1 1 1]
Exact energy: -1.0
QAOA solution: [1 1 1]
QAOA QUBO energy: -1.0
Energy match:
True


In [9]:
Q = np.array([
    [-3,  2, -1,  0,  2,  0, -2,  1],
    [ 0, -2,  3, -2,  0,  1,  0, -1],
    [ 0,  0, -4,  2, -2,  0,  1,  0],
    [ 0,  0,  0, -1,  3, -3,  0,  2],
    [ 0,  0,  0,  0, -3,  2, -1,  0],
    [ 0,  0,  0,  0,  0, -2,  2, -2],
    [ 0,  0,  0,  0,  0,  0, -3,  1],
    [ 0,  0,  0,  0,  0,  0,  0, -2],
], dtype=float)

exact_x, exact_energy = brute_force_qubo(Q)

qaoa_out = solve_qubo_with_qaoa(
        Q=Q,
        reps=3,
        maxiter=1000,
        seed=123,
    )

print("Exact solution:", exact_x)
print("Exact energy:", exact_energy)

print("QAOA solution:", qaoa_out["solution"])
print("QAOA QUBO energy:", qaoa_out["qubo_energy"])

print("Energy match:")
print(np.isclose(exact_energy, qaoa_out["qubo_energy"], atol=1e-9))

Exact solution: [1 0 1 0 1 0 1 0]
Exact energy: -16.0
QAOA solution: [1 0 1 0 1 1 1 1]
QAOA QUBO energy: -16.0
Energy match:
True
